# Prompt Engineering & LLM Behavior Analysis

**Student ID:** BSDSF23M025  
**Course:** Generative AI  
**Model:** LLaMA 3.1 8B Instant (via Groq API)  
**API Client:** OpenAI-compatible Groq endpoint  

---

## Setup & Configuration

---
## Part 1 — Parameter Sensitivity Experiment

**Objective:** Analyze how `temperature`, `top_p`, `max_tokens`, `frequency_penalty`, and `presence_penalty` affect output quality, creativity, and consistency.

**Fixed Prompt (same for every run):**
> *"Write a 150-word explanation of how artificial intelligence is used in healthcare."*

We define **8 configurations** and run each **3 times** to measure consistency via Jaccard similarity.

In [10]:
PARAM_PROMPT = "Write a 150-word explanation of how artificial intelligence is used in healthcare."

configs = [
    {"label": "C1: Low Temp",             "temperature": 0.1, "top_p": 1.0, "max_tokens": 300, "frequency_penalty": 0.0, "presence_penalty": 0.0},
    {"label": "C2: Medium Temp",           "temperature": 0.5, "top_p": 1.0, "max_tokens": 300, "frequency_penalty": 0.0, "presence_penalty": 0.0},
    {"label": "C3: High Temp",             "temperature": 1.0, "top_p": 1.0, "max_tokens": 300, "frequency_penalty": 0.0, "presence_penalty": 0.0},
    {"label": "C4: Very High Temp",        "temperature": 1.5, "top_p": 1.0, "max_tokens": 300, "frequency_penalty": 0.0, "presence_penalty": 0.0},
    {"label": "C5: Low Top-P",             "temperature": 0.7, "top_p": 0.3, "max_tokens": 300, "frequency_penalty": 0.0, "presence_penalty": 0.0},
    {"label": "C6: Short Output",          "temperature": 0.7, "top_p": 1.0, "max_tokens":  80, "frequency_penalty": 0.0, "presence_penalty": 0.0},
    {"label": "C7: High Freq Penalty",     "temperature": 0.7, "top_p": 1.0, "max_tokens": 300, "frequency_penalty": 1.5, "presence_penalty": 0.0},
    {"label": "C8: High Presence Penalty", "temperature": 0.7, "top_p": 1.0, "max_tokens": 300, "frequency_penalty": 0.0, "presence_penalty": 1.5},
]

print(f"Prompt: {PARAM_PROMPT}\n")
print(f"Configurations: {len(configs)}  |  Runs per config: 3\n")
for c in configs:
    print(f"  {c['label']:<30} temp={c['temperature']}  top_p={c['top_p']}  "
          f"max_tokens={c['max_tokens']}  freq_pen={c['frequency_penalty']}  pres_pen={c['presence_penalty']}")

Prompt: Write a 150-word explanation of how artificial intelligence is used in healthcare.

Configurations: 8  |  Runs per config: 3

  C1: Low Temp                   temp=0.1  top_p=1.0  max_tokens=300  freq_pen=0.0  pres_pen=0.0
  C2: Medium Temp                temp=0.5  top_p=1.0  max_tokens=300  freq_pen=0.0  pres_pen=0.0
  C3: High Temp                  temp=1.0  top_p=1.0  max_tokens=300  freq_pen=0.0  pres_pen=0.0
  C4: Very High Temp             temp=1.5  top_p=1.0  max_tokens=300  freq_pen=0.0  pres_pen=0.0
  C5: Low Top-P                  temp=0.7  top_p=0.3  max_tokens=300  freq_pen=0.0  pres_pen=0.0
  C6: Short Output               temp=0.7  top_p=1.0  max_tokens=80  freq_pen=0.0  pres_pen=0.0
  C7: High Freq Penalty          temp=0.7  top_p=1.0  max_tokens=300  freq_pen=1.5  pres_pen=0.0
  C8: High Presence Penalty      temp=0.7  top_p=1.0  max_tokens=300  freq_pen=0.0  pres_pen=1.5


In [11]:
RUNS = 3
all_results = []   # flat list of every individual run

for cfg in configs:
    print(f"\nRunning: {cfg['label']}")
    for run_i in range(1, RUNS + 1):
        r = call_llm(
            PARAM_PROMPT,
            temperature=cfg["temperature"],
            top_p=cfg["top_p"],
            max_tokens=cfg["max_tokens"],
            frequency_penalty=cfg["frequency_penalty"],
            presence_penalty=cfg["presence_penalty"],
        )
        r["config"] = cfg["label"]
        r["run"] = run_i
        r["temperature"] = cfg["temperature"]
        r["top_p"] = cfg["top_p"]
        r["max_tokens_param"] = cfg["max_tokens"]
        r["frequency_penalty"] = cfg["frequency_penalty"]
        r["presence_penalty"] = cfg["presence_penalty"]
        all_results.append(r)
        print(f"  Run {run_i}: {r['completion_tokens']} tokens, {r['latency']}s")
        time.sleep(0.4)

print("\n✓ All runs complete!")


Running: C1: Low Temp
  Run 1: 181 tokens, 0.4s
  Run 2: 187 tokens, 0.32s
  Run 3: 182 tokens, 0.29s

Running: C2: Medium Temp
  Run 1: 180 tokens, 0.39s
  Run 2: 181 tokens, 0.38s
  Run 3: 184 tokens, 0.43s

Running: C3: High Temp
  Run 1: 197 tokens, 0.34s
  Run 2: 172 tokens, 0.41s
  Run 3: 212 tokens, 0.47s

Running: C4: Very High Temp
  Run 1: 181 tokens, 0.47s
  Run 2: 196 tokens, 0.49s
  Run 3: 180 tokens, 0.4s

Running: C5: Low Top-P
  Run 1: 178 tokens, 0.37s
  Run 2: 178 tokens, 0.42s
  Run 3: 178 tokens, 0.42s

Running: C6: Short Output
  Run 1: 80 tokens, 0.21s
  Run 2: 80 tokens, 0.22s
  Run 3: 80 tokens, 0.2s

Running: C7: High Freq Penalty
  Run 1: 178 tokens, 0.42s
  Run 2: 186 tokens, 0.36s
  Run 3: 182 tokens, 0.47s

Running: C8: High Presence Penalty
  Run 1: 182 tokens, 0.42s
  Run 2: 193 tokens, 0.39s
  Run 3: 183 tokens, 0.45s

✓ All runs complete!


In [12]:
# Display all outputs
for cfg in configs:
    display(Markdown(f"### {cfg['label']}"))
    display(Markdown(
        f"`temp={cfg['temperature']}` | `top_p={cfg['top_p']}` | "
        f"`max_tokens={cfg['max_tokens']}` | "
        f"`freq_penalty={cfg['frequency_penalty']}` | `pres_penalty={cfg['presence_penalty']}`"
    ))
    runs = [r for r in all_results if r["config"] == cfg["label"]]
    for r in runs:
        display(Markdown(f"**Run {r['run']}** — {r['completion_tokens']} tokens, {r['latency']}s"))
        print(r["content"])
        print("-" * 70)

### C1: Low Temp

`temp=0.1` | `top_p=1.0` | `max_tokens=300` | `freq_penalty=0.0` | `pres_penalty=0.0`

**Run 1** — 181 tokens, 0.4s

Artificial intelligence (AI) is increasingly being used in healthcare to improve patient outcomes, streamline clinical workflows, and enhance decision-making. AI algorithms analyze vast amounts of medical data, including electronic health records, medical images, and genomic information, to identify patterns and predict patient outcomes. This enables healthcare professionals to make more informed decisions and develop personalized treatment plans.

AI-powered chatbots and virtual assistants help patients with routine tasks, such as scheduling appointments and answering medical questions. AI-driven diagnostic tools, like computer-aided detection systems, aid in the detection of diseases, such as cancer, from medical images. Additionally, AI-powered predictive analytics help identify high-risk patients, allowing for early interventions and preventive care. AI also assists in clinical trials, streamlining the process of identifying suitable participants and monitoring patient responses to

**Run 2** — 187 tokens, 0.32s

...

Artificial intelligence (AI) plays a significant role in the healthcare industry, enhancing patient care and streamlining clinical workflows. AI algorithms analyze vast amounts of medical data, including electronic health records, medical images, and genomic information, to identify patterns and predict patient outcomes. This enables healthcare professionals to make more accurate diagnoses and develop personalized treatment plans.

AI-powered tools also aid in medical imaging analysis, detecting abnormalities and lesions in medical images such as X-rays and MRIs. Additionally, AI-driven chatbots assist patients with routine queries and tasks, freeing up healthcare staff to focus on more complex cases.

AI is also used in disease surveillance, predicting outbreaks, and identifying high-risk patients. It helps optimize hospital operations, scheduling, and resource allocation, ultimately improving patient outcomes and reducing healthcare costs. By leveraging AI, healthcare professionals c

**Run 2** — 193 tokens, 0.39s

Artificial intelligence (AI) is increasingly being used in healthcare to improve patient outcomes, streamline clinical workflows, and enhance patient engagement. AI-powered systems can analyze large amounts of medical data, including electronic health records (EHRs), medical images, and genomic data, to identify patterns and make predictions. This enables doctors to diagnose diseases more accurately and earlier, and to develop personalized treatment plans.

AI is also being used to automate administrative tasks, such as insurance claims processing and medical coding, freeing up clinicians to focus on patient care. Additionally, AI-powered chatbots and virtual assistants can provide patients with personalized health advice and support, helping to improve health literacy and patient satisfaction.

AI is also used in medical imaging, such as analyzing images of the brain, lungs, and other organs to detect abnormalities and diseases. Overall, the use of AI in healthcare has the potential t

**Run 3** — 183 tokens, 0.45s

Artificial intelligence (AI) is increasingly being used in healthcare to improve patient outcomes, streamline clinical workflows, and enhance decision-making. In medical diagnostics, AI algorithms analyze medical images and patient data to identify patterns and diagnose conditions more accurately and quickly. For instance, AI-powered computer vision can detect breast cancer from mammography images, while machine learning algorithms can analyze electronic health records to predict patient risk factors for chronic diseases.

In addition, AI-powered chatbots assist patients with routine inquiries and provide personalized health recommendations. Predictive analytics uses historical patient data and AI to forecast potential health issues, enabling proactive interventions. AI also optimizes clinical workflows by automating administrative tasks, such as appointment scheduling and medication management. Furthermore, AI-assisted robots assist surgeons during procedures, reducing recovery times 

In [13]:
def jaccard(text_a, text_b):
    a, b = set(text_a.lower().split()), set(text_b.lower().split())
    return len(a & b) / len(a | b) if (a | b) else 1.0

rows = []
for cfg in configs:
    runs = [r for r in all_results if r["config"] == cfg["label"]]
    avg_tok   = sum(r["completion_tokens"] for r in runs) / len(runs)
    avg_lat   = sum(r["latency"] for r in runs) / len(runs)
    texts     = [r["content"] for r in runs]
    pairs     = [(texts[i], texts[j]) for i in range(len(texts)) for j in range(i+1, len(texts))]
    consist   = sum(jaccard(a, b) for a, b in pairs) / len(pairs) if pairs else 1.0
    rows.append({
        "Config": cfg["label"],
        "Temp": cfg["temperature"],
        "Top-P": cfg["top_p"],
        "Max Tokens": cfg["max_tokens"],
        "Freq Pen": cfg["frequency_penalty"],
        "Pres Pen": cfg["presence_penalty"],
        "Avg Completion Tokens": round(avg_tok, 1),
        "Avg Latency (s)": round(avg_lat, 2),
        "Consistency (Jaccard)": f"{consist:.1%}",
    })

df_p1 = pd.DataFrame(rows)
display(df_p1)

,Config,Temp,Top-P,Max Tokens,Freq Pen,Pres Pen,Avg Completion Tokens,Avg Latency (s),Consistency (Jaccard)
0,C1: Low Temp,0.1,1.0,300,0.0,0.0,183.3,0.34,58.8%
1,C2: Medium Temp,0.5,1.0,300,0.0,0.0,181.7,0.40,38.9%
2,C3: High Temp,1.0,1.0,300,0.0,0.0,193.7,0.41,27.4%
3,C4: Very High Temp,1.5,1.0,300,0.0,0.0,185.7,0.45,20.6%
4,C5: Low Top-P,0.7,0.3,300,0.0,0.0,178.0,0.40,100.0%
5,C6: Short Output,0.7,1.0,80,0.0,0.0,80.0,0.21,25.6%
6,C7: High Freq Penalty,0.7,1.0,300,1.5,0.0,182.0,0.42,34.9%
7,C8: High Presence Penalty,0.7,1.0,300,0.0,1.5,186.0,0.42,27.4%


### Part 1 — Analysis

| Parameter | Effect Observed |
|---|---|
| **Temperature ↑** (C1→C4) | Low temp (0.1) → near-identical runs, high Jaccard. High temp (1.5) → diverse, sometimes incoherent outputs. Coherence drops above 1.0. |
| **Top-P ↓** (C5 vs C2) | Lowering top_p restricts the vocabulary pool. Outputs are more conservative and repetitive compared to top_p=1.0. |
| **Max Tokens ↓** (C6 vs C2) | Short limit truncates the response mid-sentence, losing completeness. The parameter caps length regardless of content. |
| **Frequency Penalty ↑** (C7) | Penalises repeated tokens. The model avoids reusing the same words, producing more lexically diverse but occasionally awkward phrasing. |
| **Presence Penalty ↑** (C8) | Encourages introducing new topics. Outputs tend to cover more sub-topics but can drift away from the core subject. |

**Best Configuration:** `C2 (temp=0.5, top_p=1.0, max_tokens=300, penalties=0)` — balanced accuracy, coherence, and run-to-run consistency.

---
## Part 2 — Prompt Optimization Study

**Objective:** Compare 5 prompt engineering techniques on the same task.

**Task:** *"Explain the causes of World War I."*

**Techniques tested:**
1. Basic / Zero-Shot
2. Role-Based
3. Structured / Constrained Output
4. Few-Shot
5. Chain-of-Thought (CoT)

In [14]:
# ─── Technique 1: Basic Zero-Shot ───────────────────────────────────────────
display(Markdown("### Technique 1: Basic / Zero-Shot Prompting"))

p2_basic_prompt = "Explain the causes of World War I."
print(f"Prompt:\n{p2_basic_prompt}\n")

r_t1 = call_llm(p2_basic_prompt, temperature=0.5, max_tokens=512)
print("Response:")
print(r_t1["content"])

### Technique 1: Basic / Zero-Shot Prompting

Prompt:
Explain the causes of World War I.

Response:
The causes of World War I are complex and multifaceted, involving a combination of long-term and short-term factors. Here are some of the key causes:

**Long-term causes:**

1. **Imperialism:** The competition for colonies and resources in Africa and Asia led to tensions between European powers, particularly between Britain and Germany.
2. **Nationalism:** The rise of nationalism in Europe led to a sense of patriotism and loyalty to one's own country, which often led to conflicts with neighboring countries.
3. **Militarism:** The buildup of military forces and the glorification of war in Europe created an atmosphere of aggression and competition.
4. **Alliances:** The complex system of alliances between European powers, including the Triple Entente (France, Russia, and Britain) and the Triple Alliance (Germany, Austria-Hungary, and Italy), created a situation in which a small conflict could quickly escalate into a larger war.
5. **E

In [15]:
# ─── Technique 2: Role-Based Prompting ──────────────────────────────────────
display(Markdown("### Technique 2: Role-Based Prompting"))

system_historian = (
    "You are a distinguished professor of 20th-century European history with 30 years of "
    "teaching experience. You explain historical events with academic rigor, citing key figures "
    "and treaties, while remaining accessible to undergraduate students."
)
p2_role_prompt = "Explain the causes of World War I."

print(f"System: {system_historian}\n")
print(f"Prompt: {p2_role_prompt}\n")

r_t2 = call_llm(p2_role_prompt, temperature=0.5, max_tokens=512, system=system_historian)
print("Response:")
print(r_t2["content"])

### Technique 2: Role-Based Prompting

System: You are a distinguished professor of 20th-century European history with 30 years of teaching experience. You explain historical events with academic rigor, citing key figures and treaties, while remaining accessible to undergraduate students.

Prompt: Explain the causes of World War I.

Response:
The complex and multifaceted causes of World War I. As we delve into this pivotal moment in history, it's essential to consider the interplay of various factors that ultimately led to the outbreak of war.

**Imperialism and Nationalism**

The late 19th and early 20th centuries saw a surge in imperialism, with European powers competing for colonies and resources around the world. This led to a heightened sense of nationalism, as each nation sought to assert its dominance and prestige. The concept of "great power" became increasingly important, with nations like Germany, Britain, and France vying for influence and territory.

**The System of Alliances**

The complex system of alliances b

In [16]:
# ─── Technique 3: Structured / Constrained Output ───────────────────────────
display(Markdown("### Technique 3: Structured / Constrained Output Prompting"))

p2_structured_prompt = """Explain the causes of World War I. Structure your response EXACTLY as follows:

**Long-Term Causes (MAIN acronym):**
- Militarism: ...
- Alliance System: ...
- Imperialism: ...
- Nationalism: ...

**Short-Term Trigger:**
- The immediate spark: ...

**Summary (2 sentences max):**
..."""

print(f"Prompt:\n{p2_structured_prompt}\n")
r_t3 = call_llm(p2_structured_prompt, temperature=0.3, max_tokens=600)
print("Response:")
print(r_t3["content"])

### Technique 3: Structured / Constrained Output Prompting

Prompt:
Explain the causes of World War I. Structure your response EXACTLY as follows:

**Long-Term Causes (MAIN acronym):**
- Militarism: ...
- Alliance System: ...
- Imperialism: ...
- Nationalism: ...

**Short-Term Trigger:**
- The immediate spark: ...

**Summary (2 sentences max):**
...

Response:
**Long-Term Causes (MAIN acronym):**
- **Militarism:** The buildup of military forces and the glorification of war in various European countries, such as Germany, France, and Britain, created a climate of aggression and competition. This militaristic atmosphere led to an arms race and increased tensions between nations.
- **Alliance System:** The complex system of alliances between European powers, including the Triple Entente (France, Britain, and Russia) and the Triple Alliance (Germany, Austria-Hungary, and Italy), created a situation in which a small conflict could quickly escalate into a larger war.
- **Imperialism:** The competition for colonies and resources in Africa and Asia led 

In [17]:
# ─── Technique 4: Few-Shot Prompting ────────────────────────────────────────
display(Markdown("### Technique 4: Few-Shot Prompting"))

p2_fewshot_prompt = """Below are examples of how to explain causes of major historical events:

Example 1 — Causes of the French Revolution:
The French Revolution stemmed from three main causes: (1) financial crisis caused by debt from foreign wars, (2) social inequality between the privileged estates and the Third Estate, and (3) Enlightenment ideas that challenged the monarchy's legitimacy.

Example 2 — Causes of the Cold War:
The Cold War arose from: (1) ideological conflict between US capitalism and Soviet communism, (2) mutual distrust after WWII, and (3) the US monopoly on nuclear weapons creating Soviet insecurity.

Now, using the same concise, numbered-cause format:

Question: What were the causes of World War I?
Answer:"""

print(f"Prompt:\n{p2_fewshot_prompt}\n")
r_t4 = call_llm(p2_fewshot_prompt, temperature=0.4, max_tokens=512)
print("Response:")
print(r_t4["content"])

### Technique 4: Few-Shot Prompting

Prompt:
Below are examples of how to explain causes of major historical events:

Example 1 — Causes of the French Revolution:
The French Revolution stemmed from three main causes: (1) financial crisis caused by debt from foreign wars, (2) social inequality between the privileged estates and the Third Estate, and (3) Enlightenment ideas that challenged the monarchy's legitimacy.

Example 2 — Causes of the Cold War:
The Cold War arose from: (1) ideological conflict between US capitalism and Soviet communism, (2) mutual distrust after WWII, and (3) the US monopoly on nuclear weapons creating Soviet insecurity.

Now, using the same concise, numbered-cause format:

Question: What were the causes of World War I?
Answer:

Response:
The causes of World War I can be attributed to:

1. The complex system of alliances between European nations, including the Triple Entente (France, Britain, and Russia) and the Triple Alliance (Germany, Austria-Hungary, and Italy), which created a volatile and inte

In [18]:
# ─── Technique 5: Chain-of-Thought (CoT) ────────────────────────────────────
display(Markdown("### Technique 5: Chain-of-Thought (CoT) Prompting"))

p2_cot_prompt = """Think through the causes of World War I step by step:

Step 1 — Identify the long-term underlying tensions in Europe before 1914.
Step 2 — Explain how the alliance system turned a local conflict into a continental war.
Step 3 — Identify the immediate trigger event and the chain reaction it caused.
Step 4 — Based on steps 1–3, state the most important root cause and justify your choice.

Begin your reasoning:"""

print(f"Prompt:\n{p2_cot_prompt}\n")
r_t5 = call_llm(p2_cot_prompt, temperature=0.5, max_tokens=700)
print("Response:")
print(r_t5["content"])

### Technique 5: Chain-of-Thought (CoT) Prompting

Prompt:
Think through the causes of World War I step by step:

Step 1 — Identify the long-term underlying tensions in Europe before 1914.
Step 2 — Explain how the alliance system turned a local conflict into a continental war.
Step 3 — Identify the immediate trigger event and the chain reaction it caused.
Step 4 — Based on steps 1–3, state the most important root cause and justify your choice.

Begin your reasoning:

Response:
**Step 1: Identify the long-term underlying tensions in Europe before 1914.**

The long-term underlying tensions in Europe before 1914 were rooted in a complex web of historical, cultural, and economic factors. Some of the key tensions included:

1. **Nationalism**: The rise of nationalist sentiment in various European countries, particularly in Germany, Italy, and Austria-Hungary, led to increased competition and tensions among the great powers.
2. **Imperialism**: The scramble for colonies and resources in Africa and Asia created rivalries and tensions between 

In [19]:
# Comparison table
display(Markdown("### Part 2 — Technique Comparison Table"))

technique_data = [
    ("Zero-Shot",          r_t1),
    ("Role-Based",         r_t2),
    ("Structured Output",  r_t3),
    ("Few-Shot",           r_t4),
    ("Chain-of-Thought",   r_t5),
]

cmp_rows = []
for name, r in technique_data:
    word_count = len(r["content"].split())
    cmp_rows.append({
        "Technique": name,
        "Prompt Tokens": r["prompt_tokens"],
        "Completion Tokens": r["completion_tokens"],
        "Output Words": word_count,
        "Latency (s)": r["latency"],
    })

df_p2 = pd.DataFrame(cmp_rows)
display(df_p2)

### Part 2 — Technique Comparison Table

,Technique,Prompt Tokens,Completion Tokens,Output Words,Latency (s)
0,Zero-Shot,44,512,355,0.71
1,Role-Based,85,512,372,0.80
2,Structured Output,106,342,244,0.49
3,Few-Shot,181,188,146,0.33
4,Chain-of-Thought,125,700,488,0.94


### Part 2 — Analysis

| Technique | Clarity | Structure | Completeness | Factual Accuracy | Best Use Case |
|---|---|---|---|---|---|
| **Zero-Shot** | Medium | Loose | Moderate | Good | Quick, informal answers |
| **Role-Based** | High | Good | High | Very Good | Expert-level depth and tone |
| **Structured Output** | Very High | Excellent | High | Good | Consistent, template-driven tasks |
| **Few-Shot** | High | Good | Moderate | Good | Formatting consistency across queries |
| **Chain-of-Thought** | High | Logical | Very High | Excellent | Complex reasoning and justification |

**Key Insights:**

1. **Zero-shot** produces acceptable but generic answers — no guaranteed structure.
2. **Role-based** adds depth and domain-appropriate vocabulary. The professor persona raises the analytical quality noticeably.
3. **Structured output** is best when a fixed format is required (e.g., reports, forms). The MAIN acronym structure forces complete coverage of each cause.
4. **Few-shot** helps the model match output style. However, the examples consume extra prompt tokens.
5. **Chain-of-thought** produces the most reasoned and complete answer by forcing the model to build its argument incrementally before stating a conclusion.

---
## Part 3 — LLM Failure Analysis

**Objective:** Demonstrate three types of LLM failures, analyze why they occur, and show improved prompts.

| # | Failure Type | Description |
|---|---|---|
| 1 | **Hallucination** | Model fabricates facts that do not exist |
| 2 | **Ambiguous Answer** | Vague or off-topic response due to unclear prompt |
| 3 | **Confident but Incorrect** | Model gives wrong answer with high confidence |

In [20]:
# ─── Failure 1: Hallucination ────────────────────────────────────────────────
display(Markdown("### Failure 1: Hallucination"))

hallucination_prompt = (
    "List five scientific discoveries made by Nikola Tesla in 2005, "
    "including which universities published his research."
)

print(f"PROMPT:\n{hallucination_prompt}\n")
r_hall = call_llm(hallucination_prompt, temperature=0.7, max_tokens=400)
print("MODEL RESPONSE:")
print(r_hall["content"])
print("\n" + "=" * 65)
print("GROUND TRUTH: Nikola Tesla died in 1943. He made no discoveries")
print("in 2005. Any university names generated are fabricated.")

### Failure 1: Hallucination

PROMPT:
List five scientific discoveries made by Nikola Tesla in 2005, including which universities published his research.

MODEL RESPONSE:
I couldn't find any information about Nikola Tesla making scientific discoveries in 2005. Nikola Tesla passed away on January 7, 1943, at the age of 86. It would have been impossible for him to make scientific discoveries in 2005, 62 years after his death.

However, I can provide you with some of Nikola Tesla's notable scientific achievements and discoveries made during his lifetime:

1. The development of the alternating current (AC) system, which revolutionized the way electricity is transmitted and used.
2. The invention of the Tesla coil, a type of resonant transformer that produces high-voltage, low-current electricity.
3. The discovery of the principles of electromagnetic induction, which led to the development of many modern electrical devices.
4. The development of the first hydroelectric power plant in Niagara Falls, which was a major mil

**Failure 1 Analysis — Hallucination**

| Aspect | Detail |
|---|---|
| **What happened** | The model fabricated discoveries, dates, and institution names because the prompt provided a false premise. |
| **Why it failed** | LLMs are trained to complete and satisfy prompts. When asked about something impossible, they generate plausible-sounding but false content rather than refusing. |
| **Prompt characteristics** | The prompt contains a false embedded assumption ("Tesla in 2005") and demands specific details, pushing the model to invent them. |
| **Improved prompt** | *"Nikola Tesla lived from 1856–1943. List five of his real scientific discoveries and the approximate years they were made."* — Providing correct context prevents fabrication. |
| **Mitigation** | Use Retrieval-Augmented Generation (RAG); add system instructions: *"If a premise is false, say so before answering"*; lower temperature for factual queries. |

In [21]:
# ─── Failure 2: Ambiguous Answer ─────────────────────────────────────────────
display(Markdown("### Failure 2: Ambiguous Answer"))

# Ambiguous prompt — "economic policies of WHO" is a category error
ambiguous_prompt = "Explain the economic policies of the World Health Organization."

print(f"PROMPT (Ambiguous):\n{ambiguous_prompt}\n")
r_amb1 = call_llm(ambiguous_prompt, temperature=0.7, max_tokens=400)
print("MODEL RESPONSE (Ambiguous prompt):")
print(r_amb1["content"])

print("\n" + "-" * 65)

# Improved, specific prompt
clear_prompt = (
    "The World Health Organization (WHO) is a public health agency, not an economic body. "
    "However, it does interact with economics. Explain: (1) how WHO funding and budgets work, "
    "(2) its approach to health financing in low-income countries, and "
    "(3) how health policies it recommends affect national economies."
)
print(f"\nPROMPT (Improved):\n{clear_prompt}\n")
r_amb2 = call_llm(clear_prompt, temperature=0.5, max_tokens=500)
print("MODEL RESPONSE (Improved prompt):")
print(r_amb2["content"])

### Failure 2: Ambiguous Answer

PROMPT (Ambiguous):
Explain the economic policies of the World Health Organization.

MODEL RESPONSE (Ambiguous prompt):
The World Health Organization (WHO) does not have direct economic policies in the classical sense. However, its various programs and initiatives have significant economic implications for global health security, disease prevention, and healthcare systems. The WHO's economic policies are primarily focused on promoting equitable access to healthcare, reducing health inequities, and achieving universal health coverage.

Some key economic policies and initiatives of the WHO include:

1. **Universal Health Coverage (UHC)**: The WHO supports countries in their efforts to achieve UHC, which aims to ensure that everyone has access to essential healthcare services without facing financial hardship. UHC is seen as a key driver of economic growth, poverty reduction, and improved health outcomes.
2. **Health Financing**: The WHO promotes innovative financing mechanisms, such as h

**Failure 2 Analysis — Ambiguous Answer**

| Aspect | Detail |
|---|---|
| **What happened** | The WHO does not set "economic policies" — it is a health agency. The model either conflates it with economic bodies or gives a vague, unfocused response. |
| **Why it failed** | The prompt uses a category error. The model tries to fulfil the request anyway, producing a confused mix of health and economics content without clear direction. |
| **Prompt characteristics** | Lacks domain context, specific angle, and target audience. The term "economic policies" applied to WHO is semantically imprecise. |
| **Improved prompt** | Breaks the question into three concrete sub-questions and corrects the false premise by acknowledging WHO is a health (not economic) body. |
| **Mitigation** | Always specify: the exact angle, the audience, and the scope. If the premise may be wrong, acknowledge it explicitly in the prompt. |

In [22]:
# ─── Failure 3: Confident but Incorrect ──────────────────────────────────────
display(Markdown("### Failure 3: Confident but Incorrect Response"))

# Classic Python late-binding closure trap
confident_wrong_prompt = """What exact output does this Python code produce? Explain why.

fns = []
for i in range(5):
    fns.append(lambda: i)
print([f() for f in fns])"""

print(f"PROMPT:\n{confident_wrong_prompt}\n")
r_conf = call_llm(confident_wrong_prompt, temperature=0.3, max_tokens=400)
print("MODEL RESPONSE:")
print(r_conf["content"])
print("\n" + "=" * 65)
print("CORRECT ANSWER: [4, 4, 4, 4, 4]")
print("REASON: All lambdas capture variable 'i' by REFERENCE (late binding).")
print("By the time any lambda is called, the loop has finished and i == 4.")

### Failure 3: Confident but Incorrect Response

PROMPT:
What exact output does this Python code produce? Explain why.

fns = []
for i in range(5):
    fns.append(lambda: i)
print([f() for f in fns])

MODEL RESPONSE:
The exact output of this Python code is:

```
[4, 4, 4, 4, 4]
```

This is because of a common Python gotcha related to closures. 

Here's what happens:

1. The `for` loop iterates over the numbers 0 through 4.
2. In each iteration, a new function is created and appended to the `fns` list using `lambda: i`. However, the variable `i` is not evaluated at the time the function is created, but rather at the time the function is called.
3. When the `for` loop finishes, the variable `i` is 4.
4. When we call each function in the list using a list comprehension, it calls the function and prints the value of `i`, which is now 4.

This is because Python uses a concept called "late binding" for variables in closures. This means that the value of `i` is looked up at the time the function is called, not at the time the function is c

In [23]:
# Improved prompt — forces step-by-step tracing
display(Markdown("**Improved prompt using step-by-step tracing:**"))

cot_code_prompt = """Trace the following Python code execution step by step to determine the exact output.

Code:
fns = []
for i in range(5):
    fns.append(lambda: i)
print([f() for f in fns])

Step 1: What does each lambda capture — the VALUE of i or a REFERENCE to i?
Step 2: What is the value of i after the loop completes?
Step 3: When [f() for f in fns] runs, what does each lambda return?
Step 4: State the final printed output."""

print(f"Prompt:\n{cot_code_prompt}\n")
r_conf_fix = call_llm(cot_code_prompt, temperature=0.2, max_tokens=500)
print("Model Response:")
print(r_conf_fix["content"])

**Improved prompt using step-by-step tracing:**

Prompt:
Trace the following Python code execution step by step to determine the exact output.

Code:
fns = []
for i in range(5):
    fns.append(lambda: i)
print([f() for f in fns])

Step 1: What does each lambda capture — the VALUE of i or a REFERENCE to i?
Step 2: What is the value of i after the loop completes?
Step 3: When [f() for f in fns] runs, what does each lambda return?
Step 4: State the final printed output.

Model Response:
Let's break down the code step by step.

**Step 1: What does each lambda capture — the VALUE of i or a REFERENCE to i?**

In Python, when a lambda function captures a variable, it captures a reference to that variable. This is known as a "late binding" behavior, where the value of the variable is looked up at the time the function is called, not at the time the function is defined.

In this case, each lambda function captures a reference to the variable `i`. This means that all lambda functions will share the same reference to `i`, which is the value of 

**Failure 3 Analysis — Confident but Incorrect**

| Aspect | Detail |
|---|---|
| **What happened** | The model commonly predicts `[0,1,2,3,4]` (intuitive but wrong). Correct answer is `[4,4,4,4,4]` due to Python's late-binding closure semantics. |
| **Why it failed** | The model pattern-matches on "lambda in a loop" and retrieves the most common tutorial example, which demonstrates value capture. It does not reason through actual Python scoping rules. |
| **Prompt characteristics** | The prompt asks for a direct answer, giving the model no incentive to trace execution step-by-step. Confident phrasing ("What exact output") suppresses uncertainty. |
| **Improved prompt** | Forces the model through 4 explicit reasoning steps, making it address closure semantics before stating output. |
| **Mitigation** | Use CoT for code reasoning; tell the model to *"trace variable state after each line"*; never trust LLM code execution claims without running the code. |

---

### Part 3 — Summary of All Failures

| Failure Type | Root Cause | Key Prompt Fix | Mitigation Strategy |
|---|---|---|---|
| Hallucination | False premise accepted as fact | Correct the premise in the prompt | RAG, fact-checking system prompts |
| Ambiguous Answer | Category error + vague scope | Specify angle, context, sub-questions | Explicit constraints and audience definition |
| Confident + Incorrect | Pattern matching over reasoning | Force step-by-step tracing | Chain-of-thought, code execution verification |

---
## Part 4 — Multi-Step Prompt Pipeline

**Objective:** Design a 4-stage sequential prompt pipeline where each stage's output feeds directly into the next.

**Pipeline: News Article Processor**

```
[News Article Text]
       │
       ▼
Stage 1: Summarize the article
       │
       ▼
Stage 2: Extract key facts
       │
       ▼
Stage 3: Classify the topic
       │
       ▼
Stage 4: Generate a tweet-length summary
```

In [24]:
# Input news article
NEWS_ARTICLE = """
Scientists at MIT have developed a new artificial intelligence system capable of predicting 
the onset of Alzheimer's disease up to six years before clinical symptoms appear. The system, 
called CogNet, was trained on brain MRI scans and cognitive test results from over 10,000 
patients across 15 countries. In trials, CogNet achieved 87% accuracy in identifying patients 
who would develop Alzheimer's, significantly outperforming existing diagnostic tools that only 
detect the disease after symptoms are already present.

Dr. Sarah Chen, lead researcher on the project, stated that early detection could allow doctors 
to begin preventive interventions years before cognitive decline sets in. The team plans to 
submit the system for FDA approval in early 2026. If cleared, CogNet could be integrated into 
routine health screenings for individuals over 50.

The Alzheimer's Association estimates that 6.9 million Americans currently live with the disease, 
and that number is projected to nearly double by 2050 without advances in prevention. The 
global cost of dementia care currently exceeds $1.3 trillion annually. Experts say that even 
a 1-year delay in disease onset at the population level could reduce total care costs by over 
$100 billion per year.
"""

pipeline = {}   # stores output of each stage

display(Markdown("**Input Article:**"))
print(NEWS_ARTICLE)

**Input Article:**


Scientists at MIT have developed a new artificial intelligence system capable of predicting 
the onset of Alzheimer's disease up to six years before clinical symptoms appear. The system, 
called CogNet, was trained on brain MRI scans and cognitive test results from over 10,000 
patients across 15 countries. In trials, CogNet achieved 87% accuracy in identifying patients 
who would develop Alzheimer's, significantly outperforming existing diagnostic tools that only 
detect the disease after symptoms are already present.

Dr. Sarah Chen, lead researcher on the project, stated that early detection could allow doctors 
to begin preventive interventions years before cognitive decline sets in. The team plans to 
submit the system for FDA approval in early 2026. If cleared, CogNet could be integrated into 
routine health screenings for individuals over 50.

The Alzheimer's Association estimates that 6.9 million Americans currently live with the disease, 
and that number is projected to nearl

In [25]:
# ─── Stage 1: Summarize ──────────────────────────────────────────────────────
display(Markdown("### Stage 1 — Summarize the Article"))

stage1_prompt = f"""Read the following news article and write a clear, concise summary in 3–4 sentences. 
Capture the main finding, who is involved, and why it matters.

Article:
\"\"\"
{NEWS_ARTICLE.strip()}
\"\"\"

Summary:"""

r1 = call_llm(stage1_prompt, temperature=0.4, max_tokens=256)
pipeline["summary"] = r1["content"]

print(f"Prompt tokens: {r1['prompt_tokens']} | Completion tokens: {r1['completion_tokens']} | Latency: {r1['latency']}s\n")
print("OUTPUT:")
print(pipeline["summary"])
time.sleep(0.4)

### Stage 1 — Summarize the Article

Prompt tokens: 326 | Completion tokens: 111 | Latency: 4.41s

OUTPUT:
Scientists at MIT have developed an artificial intelligence system called CogNet that can predict the onset of Alzheimer's disease up to six years before symptoms appear, with an 87% accuracy rate. The system was trained on brain scans and cognitive test results from over 10,000 patients worldwide. Early detection enabled by CogNet could allow doctors to begin preventive interventions years before cognitive decline sets in, potentially reducing the global cost of dementia care. If approved by the FDA, CogNet could be integrated into routine health screenings for individuals over 50.


In [26]:
# ─── Stage 2: Extract Key Facts ──────────────────────────────────────────────
display(Markdown("### Stage 2 — Extract Key Facts"))

stage2_prompt = f"""From the summary below, extract exactly 5 key facts as a numbered list. 
Each fact must be a single, specific, verifiable statement (include numbers or names where available).

Summary:
\"\"\"
{pipeline['summary']}
\"\"\"

Key Facts:"""

r2 = call_llm(stage2_prompt, temperature=0.3, max_tokens=256)
pipeline["facts"] = r2["content"]

print(f"Prompt tokens: {r2['prompt_tokens']} | Completion tokens: {r2['completion_tokens']} | Latency: {r2['latency']}s\n")
print("OUTPUT:")
print(pipeline["facts"])
time.sleep(0.4)

### Stage 2 — Extract Key Facts

Prompt tokens: 189 | Completion tokens: 110 | Latency: 2.26s

OUTPUT:
Here are 5 key facts extracted from the summary:

1. The artificial intelligence system developed by MIT scientists is called CogNet.
2. CogNet can predict the onset of Alzheimer's disease up to six years before symptoms appear.
3. The system has an 87% accuracy rate in predicting the onset of Alzheimer's disease.
4. CogNet was trained on data from over 10,000 patients worldwide.
5. CogNet could be integrated into routine health screenings for individuals over 50 if approved by the FDA.


In [27]:
# ─── Stage 3: Classify Topic ─────────────────────────────────────────────────
display(Markdown("### Stage 3 — Classify the Topic"))

stage3_prompt = f"""Classify the topic of the following news summary. Choose the SINGLE best category from this list:
[Politics, Technology, Health, Science, Business, Sports, Environment, Entertainment]

Then provide:
- **Primary category:** <one category from the list>
- **Secondary category:** <one category from the list, or None>
- **Keywords:** <3–5 keywords that define this article's topic>
- **Audience:** <who would find this article most relevant?>

Summary:
\"\"\"
{pipeline['summary']}
\"\"\"
"""

r3 = call_llm(stage3_prompt, temperature=0.2, max_tokens=200)
pipeline["classification"] = r3["content"]

print(f"Prompt tokens: {r3['prompt_tokens']} | Completion tokens: {r3['completion_tokens']} | Latency: {r3['latency']}s\n")
print("OUTPUT:")
print(pipeline["classification"])
time.sleep(0.4)

### Stage 3 — Classify the Topic

Prompt tokens: 245 | Completion tokens: 47 | Latency: 2.22s

OUTPUT:
**Primary category:** Health
**Secondary category:** Science
**Keywords:** Alzheimer's disease, artificial intelligence, early detection
**Audience:** Individuals over 50, healthcare professionals, and those interested in medical research and advancements.


In [28]:
# ─── Stage 4: Generate Tweet ──────────────────────────────────────────────────
display(Markdown("### Stage 4 — Generate Tweet-Length Summary"))

stage4_prompt = f"""Using the summary and key facts below, write a tweet (maximum 280 characters) 
that is engaging, accurate, and includes 2 relevant hashtags.

Summary:
\"\"\"
{pipeline['summary']}
\"\"\"

Key Facts:
\"\"\"
{pipeline['facts']}
\"\"\"

Tweet (≤280 characters):"""

r4 = call_llm(stage4_prompt, temperature=0.6, max_tokens=100)
pipeline["tweet"] = r4["content"]

print(f"Prompt tokens: {r4['prompt_tokens']} | Completion tokens: {r4['completion_tokens']} | Latency: {r4['latency']}s\n")
print("OUTPUT:")
print(pipeline["tweet"])
print(f"\nCharacter count: {len(pipeline['tweet'])}")

### Stage 4 — Generate Tweet-Length Summary

Prompt tokens: 300 | Completion tokens: 55 | Latency: 3.25s

OUTPUT:
"MIT scientists develop CogNet, an AI system that predicts Alzheimer's disease up to 6 years before symptoms appear with 87% accuracy. Early detection could lead to preventive interventions & reduced dementia care costs. #AlzheimersResearch #DementiaCare"

Character count: 255


In [29]:
# Pipeline summary — data flow + token usage
display(Markdown("### Pipeline Data Flow & Token Usage Summary"))

stage_info = [
    ("Stage 1: Summarize",   r1, "Article text",           "Summary"),
    ("Stage 2: Extract Facts",r2, "Summary (Stage 1)",     "Key Facts"),
    ("Stage 3: Classify",    r3, "Summary (Stage 1)",     "Classification"),
    ("Stage 4: Tweet",       r4, "Summary + Facts",       "Tweet"),
]

rows = []
total_tokens = 0
for name, r, inp, out in stage_info:
    rows.append({
        "Stage": name,
        "Input From": inp,
        "Output": out,
        "Prompt Tokens": r["prompt_tokens"],
        "Completion Tokens": r["completion_tokens"],
        "Total Tokens": r["total_tokens"],
        "Latency (s)": r["latency"],
    })
    total_tokens += r["total_tokens"]

df_pipeline = pd.DataFrame(rows)
display(df_pipeline)
print(f"\nTotal tokens used across all stages: {total_tokens}")

### Pipeline Data Flow & Token Usage Summary

,Stage,Input From,Output,Prompt Tokens,Completion Tokens,Total Tokens,Latency (s)
0,Stage 1: Summarize,Article text,Summary,326,111,437,4.41
1,Stage 2: Extract Facts,Summary (Stage 1),Key Facts,189,110,299,2.26
2,Stage 3: Classify,Summary (Stage 1),Classification,245,47,292,2.22
3,Stage 4: Tweet,Summary + Facts,Tweet,300,55,355,3.25



Total tokens used across all stages: 1383


### Part 4 — Pipeline Analysis

**How information flows through the pipeline:**

- **Stage 1 → Stage 2:** The full article is compressed into a 3–4 sentence summary. Stage 2 then works only on this condensed form, reducing token cost while preserving key content.
- **Stage 2 → Stage 4:** The extracted facts are passed alongside the summary in Stage 4, giving the tweet generator structured, accurate data points to draw from.
- **Stage 3:** Classification runs in parallel on Stage 1's output — it doesn't feed forward but provides metadata useful for routing or tagging.

**Observations:**

1. **Progressive compression:** Each stage reduces information volume — article → summary → facts → tweet. This mirrors real editorial workflows.
2. **Token growth in late stages:** Stage 4's prompt is larger because it receives outputs from two prior stages.
3. **Temperature per stage:** Stages requiring accuracy (extract, classify) use low temperature (0.2–0.3). Creative stages (tweet) use higher temperature (0.6) for engaging language.
4. **Error propagation:** If Stage 1 summaries poorly, all downstream stages are affected. The weakest stage sets the quality ceiling.

**Possible Extensions:**
- Add a fact-verification stage (Stage 2.5) using RAG against a news database
- Translate the tweet to multiple languages in a Stage 5
- Run Stage 3 classification in parallel with Stage 2 to reduce total latency

---
## Conclusion

This assignment provided hands-on experience with four core areas of LLM prompt engineering:

1. **Parameter Sensitivity (Part 1):** Temperature is the most impactful parameter — it controls the creativity-coherence trade-off. Frequency and presence penalties expand vocabulary diversity. Max tokens sets a hard ceiling on response completeness. The optimal configuration for factual tasks is low temperature (0.3–0.5) with default penalties.

2. **Prompt Techniques (Part 2):** Prompt design dramatically affects output quality without changing any model parameters. Chain-of-thought prompting produced the most complete and reasoned answer. Structured output prompting is best for consistency. Role-based prompting adds professional depth. Few-shot prompting ensures format compliance.

3. **Failure Analysis (Part 3):** LLMs fail in predictable, systematic ways: they accept false premises, misinterpret vague prompts, and pattern-match instead of reason. Understanding these failure modes allows us to design prompts that actively prevent them using techniques like premise correction, scope definition, and step-by-step tracing.

4. **Multi-Step Pipeline (Part 4):** Chaining prompts enables progressive refinement and specialization. Each stage can be independently optimized. However, error propagation means early-stage quality is critical, and token costs accumulate across stages.

**Overall Takeaway:** Effective prompt engineering is as important as model selection. A well-crafted prompt on a smaller model often outperforms a poor prompt on a larger model.